# Session 22: Language models and building with them

You build the search half of retrieval-augmented generation (RAG) over the eight Lakeside Mart policy files, then score it with an evaluation set. Rhythm for every step: **predict**, run, **read** the output, **change** one thing. A line that starts with OK means your number matches. Data is fictional. Never paste real customer data into a tool you do not control.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, re, zipfile, urllib.request
DATA_LINK = "https://drive.google.com/file/d/11Z5tr2iKuPROn5cyi9P7me6nd3naKfNp/view?usp=sharing"   # the Google Drive link to data.zip that your tutor shares (leave empty to upload data.zip yourself)
if not os.path.exists("orders.csv"):
    if DATA_LINK:
        m = re.search(r"/d/([A-Za-z0-9_-]+)|[?&]id=([A-Za-z0-9_-]+)", DATA_LINK)
        file_id = (m.group(1) or m.group(2)) if m else DATA_LINK
        urllib.request.urlretrieve("https://drive.google.com/uc?export=download&id=" + file_id, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data files ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

In [ ]:
# Automatic check helper. Run this cell once. You do not need to read it.
_results = {}   # label -> True or False; running a check again replaces its old result
def check(label, value, expected, tol=0.01):
    try:
        import numpy as _np
        ok = bool(_np.all(_np.abs(_np.asarray(value, dtype=float) - _np.asarray(expected, dtype=float)) <= tol))
    except Exception:
        ok = False
    _results[label] = ok
    print(("OK       " if ok else "NOT YET  ") + label)
def show_checks():
    n = len(_results); k = sum(_results.values())
    print("All %d checks pass" % n if k == n and n else "%d of %d checks pass" % (k, n))

In [ ]:
# Helper: returns nan instead of stopping when your answer is not ready yet. You do not need to read it.
def safe(f):
    try:
        return f()
    except Exception:
        return float("nan")

---
## Step 1: Chunking

Split each policy file into chunks: here, one sentence per chunk. Keep the file name with every chunk.

In [ ]:
import os, re, sys
import numpy as np
import pandas as pd

os.makedirs('keep', exist_ok=True)   # the folder where you save what you keep

**PREDICT first.** Eight short files of about 90 words each. How many sentence chunks will you get: about 10, 60 or 600?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
chunks = []
for name in sorted(os.listdir('policies')):
    text = ' '.join(open('policies/' + name).read().split())
    pieces = [text]  # your code here: re.split(pattern, text): split after a full stop that is followed by a capital letter
    chunks += [(name, p) for p in pieces]

In [ ]:
n_words = sum(len(p.split()) for _, p in chunks)
print(len(chunks), 'chunks,', n_words, 'words')
check('number of chunks', len(chunks), 59, 0)
check('words in all chunks', n_words, 736, 0)

**CHANGE one thing.** Print `chunks[28]`. Which file and sentence is it? Then think: what would change if a chunk were two sentences?

> What I saw: 

---
## Step 2: Vectors for the chunks: bag of words and TF-IDF

The same recipe as Session 21, now on chunks. Rows are chunks and columns are words.

In [ ]:
STOP = set(['a', 'am', 'an', 'and', 'are', 'as', 'at', 'be', 'been', 'by', 'can', 'could', 'did', 'do', 'does', 'for', 'from', 'had', 'has', 'have', 'how', 'i', 'in', 'is', 'it', 'its', 'me', 'my', 'of', 'on', 'or', 'our', 'that', 'the', 'this', 'to', 'very', 'was', 'we', 'were', 'what', 'when', 'which', 'will', 'with', 'would', 'you', 'your'])

def tokens(text):
    return [w for w in re.findall('[a-z0-9]+', text.lower()) if w not in STOP]

docs = [tokens(p) for _, p in chunks]
vocab = sorted({w for d in docs for w in d})
index = {w: i for i, w in enumerate(vocab)}
X = np.array([[d.count(w) for w in vocab] for d in docs])
print(X.shape)

In [ ]:
df = None  # number of chunks containing each word: (X > 0).sum(axis=0)

In [ ]:
idf = None  # inverse document frequency: np.log(number of chunks / df)

In [ ]:
try:
    T = X * idf
    T = T / np.maximum(np.linalg.norm(T, axis=1, keepdims=True), 1e-9)   # unit length rows
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
def vec(text):
    v = np.zeros(len(vocab))
    for w in tokens(text):
        if w in index:
            v[index[w]] += 1
    v = v * idf
    n = np.linalg.norm(v)
    return v / n if n else v

In [ ]:
check('vocabulary size', len(vocab), 293, 0)
check('idf of the word points', safe(lambda: idf[index['points']]), 2.6912, 0.001)

---
## Step 3: Retrieval and ranking

Score every chunk against the question with cosine similarity, sort the scores, and keep the top k.

**PREDICT first.** For "How many days do I have to return an item?", which file will the top chunk come from?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
def retrieve(question, k=3):
    return None  # scores = T @ vec(question); then the indices of the k highest scores

In [ ]:
Q = 'How many days do I have to return an item?'

In [ ]:
try:
    top = retrieve(Q)
    for i in top:
        print(round(float(T[i] @ vec(Q)), 2), chunks[i][0], '|', chunks[i][1][:60])
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
check('top chunk is from returns_policy.txt', safe(lambda: chunks[top[0]][0] == 'returns_policy.txt'), True, 0)
check('score of the top chunk', safe(lambda: float(T[top[0]] @ vec(Q))), 0.5639, 0.01)

**READ the output.** The second and third chunks are wrong. Which words did they share with the question?

> My answer: 

---
## Step 4: The evaluation set

A labelled test set: each question has the exact answer text. A chunk is relevant when it contains that text.

In [ ]:
tests = pd.DataFrame([
    ('How long do points last?', 'loyalty_programme.txt', 'Points expire after 12 months'),
    ('How many days do I have to return an item?', 'returns_policy.txt', 'within 7 days of purchase'),
    ('How long do mobile money refunds take?', 'returns_policy.txt', 'up to 3 working days'),
    ('Which towns does delivery not reach?', 'delivery.txt', 'Gulu, Mbale, Mbarara or Fort Portal'),
    ('What is the delivery fee within 5 kilometres?', 'delivery.txt', 'UGX 5,000 within 5 kilometres'),
    ('What percent discount do Gold customers get?', 'loyalty_programme.txt', 'Gold customers receive 5 percent off'),
    ('How long is customer data kept?', 'data_privacy.txt', '3 years after the last purchase'),
    ('What temperature must dairy be stored at?', 'food_safety.txt', 'below 5 degrees Celsius'),
    ('When do branches open on Sunday?', 'opening_hours.txt', 'from 9 am to 6 pm on Sunday'),
    ('What happens when more than 5 percent of a delivery is damaged?', 'supplier_complaints.txt', 'the delivery is rejected'),
    ('Does Lakeside Mart ask for my PIN?', 'mobile_money.txt', "never asks for a customer's PIN"),
    ('Can points be exchanged for cash?', 'loyalty_programme.txt', 'Points cannot be exchanged for cash'),
], columns=['question', 'policy', 'answer'])
print(len(tests), 'questions')

In [ ]:
relevant = None  # for each answer, the set of chunk indices whose text contains it: {i for i, (_, s) in enumerate(chunks) if a in s}

In [ ]:
n_relevant = sum(len(r) for r in relevant) if relevant is not None else 0
print(n_relevant, 'relevant chunks for', len(tests), 'questions')
check('one relevant chunk per question', n_relevant, 12, 0)

**CHANGE one thing.** Add a 13th question of your own: copy a sentence from any policy file as the answer. Run the whole evaluation again later and see how the scores move.

> What I saw: 

---
## Step 5: Precision at k and recall at k

Precision at k = relevant in the top k divided by k. Recall at k = relevant in the top k divided by all relevant chunks.

**PREDICT first.** Only one chunk is right per question. What is the largest possible precision at 5? And will recall at 5 be higher or lower than recall at 1?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
def precision_at_k(top, rel, k):
    return None  # relevant chunks in the top k, divided by k

In [ ]:
def recall_at_k(top, rel, k):
    return None  # relevant chunks in the top k, divided by all relevant chunks

In [ ]:
try:
    def evaluate(retriever, k):
        p = [precision_at_k(retriever(q, 5), rel, k) for q, rel in zip(tests.question, relevant)]
        r = [recall_at_k(retriever(q, 5), rel, k) for q, rel in zip(tests.question, relevant)]
        return float(np.mean(p)), float(np.mean(r))

    for k in (1, 3, 5):
        print('k =', k, 'precision, recall:', [round(v, 3) for v in evaluate(retrieve, k)])
    p1, r1 = evaluate(retrieve, 1)
    p3, r3 = evaluate(retrieve, 3)
    p5, r5 = evaluate(retrieve, 5)
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
check('mean precision at 1', safe(lambda: p1), 0.8333, 0.002)
check('mean recall at 1', safe(lambda: r1), 0.8333, 0.002)
check('mean precision at 3', safe(lambda: p3), 0.3056, 0.002)
check('mean recall at 3', safe(lambda: r3), 0.9167, 0.002)
check('mean recall at 5', safe(lambda: r5), 1.0000, 0.002)

**READ the output.** Why does precision fall as k grows while recall rises?

> My answer: 

---
## Step 6: Read the failures

Find the questions where the right chunk is not ranked first, and see what the search put first instead.

In [ ]:
def rank_of_right(question, rel):
    return None  # position (1 is first) of the relevant chunk in the full ranking of all chunks

In [ ]:
try:
    tests['rank_of_right'] = [rank_of_right(q, rel) for q, rel in zip(tests.question, relevant)]
    n_fail = int((tests.rank_of_right > 1).sum())
    print(n_fail, 'questions have the right chunk below rank 1')
    tests[tests.rank_of_right > 1][['question', 'rank_of_right']]
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
check('questions whose right chunk is not first', safe(lambda: n_fail), 2, 0)
check('rank of the right chunk for the points question', safe(lambda: tests.rank_of_right.iloc[0]), 4, 0)

**READ the output.** For "How long do points last?" print the top 3 chunks. Why does the word last cause trouble? Write one sentence.

> My answer: 

---
## Step 7: Change the method and score it again

Replace TF-IDF with plain word overlap: the score is the number of question words found in the chunk. Same evaluation set, same scores.

In [ ]:
def retrieve_overlap(question, k=3):
    q = set(tokens(question))
    scores = np.array([len(q & set(d)) for d in docs], float)
    return np.argsort(-scores, kind='stable')[:k]

In [ ]:
p3_overlap = None  # evaluate(retrieve_overlap, 3) returns precision and recall; keep the precision

In [ ]:
print(safe(lambda: round(p3_overlap, 3)))
check('mean precision at 3 with word overlap', safe(lambda: p3_overlap), 0.3333, 0.002)

**READ the output.** Is the simpler method worse? Twelve questions are too few to prefer one method. Write what you would test next before choosing.

> My answer: 

---
## Step 8: Grounding: build the prompt

Put the retrieved chunks in the prompt and tell the model to answer only from them.

In [ ]:
SYSTEM_PROMPT = 'Answer only from the passages below. If the answer is not in the passages, say: I could not find that in the policies.'

In [ ]:
def build_prompt(question, passages):
    return None  # system prompt, then 'Passages:', the numbered passages, then the question and 'Answer:'

In [ ]:
try:
    prompt = build_prompt(Q, [chunks[i][1] for i in retrieve(Q, 3)])
    print(prompt)
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
check('words in the grounded prompt', safe(lambda: len(prompt.split())), 95, 0)
check('prompt contains the returns sentence', safe(lambda: 'within 7 days of purchase' in prompt), True, 0)

**CHANGE one thing.** Set k to 1 and build the prompt again. Count the words. What is the risk of k = 1?

> What I saw: 

---
## Step 9: Optional: a small open model in Colab

Colab only. The model is SmolLM2 360M Instruct (Apache 2.0 licence on its model card). Read the card before you use any model at work. Small models can be wrong or ignore your rules, so read the answer against the source chunk. Use a model that is not gated (no account or access request). Generating 60 tokens on the free CPU may take a minute.

In [ ]:
MODEL_NAME = 'HuggingFaceTB/SmolLM2-360M-Instruct'   # Apache 2.0 licence, see the model card
RUN_MODEL = 'google.colab' in sys.modules
if RUN_MODEL and isinstance(prompt, str):
    from transformers import pipeline
    generator = pipeline('text-generation', model=MODEL_NAME)
    answer = generator(prompt, max_new_tokens=60, do_sample=False)[0]['generated_text'][len(prompt):]
    print(answer)
else:
    print('This step runs in Colab. Skipping.')

**READ the output.** Does the answer match the source chunk? Write the answer, your check and your decision in your audit log (next step).

> My answer: 

**CHANGE one thing.** Try the chat format: build `messages = [{"role": "user", "content": prompt}]` and pass it to the generator instead of the raw string. Does the answer follow the rules better?

> What I saw: 

---
## Step 10: Keep: an evaluation file and an audit log

Save your scores question by question, and record what you asked an assistant (or the model) and how you checked it.

In [ ]:
try:
    eval_results = tests[['question', 'policy', 'rank_of_right']].copy()
    eval_results['top1_correct'] = eval_results.rank_of_right == 1
    eval_results['precision_at_3'] = [precision_at_k(retrieve(q, 5), rel, 3) for q, rel in zip(tests.question, relevant)]
    eval_results['recall_at_3'] = [recall_at_k(retrieve(q, 5), rel, 3) for q, rel in zip(tests.question, relevant)]
    eval_results.to_csv('keep/eval_results.csv', index=False)
    print(len(eval_results), 'rows saved')
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
audit_log = pd.DataFrame(columns=['what_i_asked', 'what_it_gave', 'how_i_checked', 'decision'])
# Your turn: add one row, for example audit_log.loc[len(audit_log)] = ['what you asked', 'what it gave', 'how you checked it', 'your decision']
audit_log.to_csv('keep/audit_log.csv', index=False)
audit_log

In [ ]:
check('eval_results has 12 rows', safe(lambda: len(eval_results)), 12, 0)
check('audit log has at least one row', min(len(audit_log), 1), 1, 0)
check('audit log has 4 columns', audit_log.shape[1], 4, 0)
show_checks()

**My finding (three sentences):** 1) precision at 1 and recall at 3 of my retrieval, 2) one failure and why, 3) one thing I would change before putting this in front of customers.

> 